In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import ResNet50, InceptionResNetV2
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.resnet50 import preprocess_input
from sklearn import svm
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.layers import Input

# Load pre-trained ResNet50 model
#nput_tensor = Input(shape=(400, 320, 3))
resnet50base_model = ResNet50(weights='imagenet', include_top=False)
resnet50model = tf.keras.Model(inputs=resnet50base_model.input, outputs=tf.keras.layers.GlobalAveragePooling2D()(resnet50base_model.output))

# Function to extract features from an image
def extract_features(img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = preprocess_input(x)
    features = resnet50base_model.predict(x)
    return features.flatten()



In [ ]:
import glob, os
 
xtrain = []

path = "/path/to/private-data/Croped_Pictures/train/Left/*"
for directory_path in glob.glob(path):
    label=directory_path.split("\\")[-1]
    for image_path in glob.glob(os.path.join(directory_path,"*.bmp")):

        if label == "Inlet (5 marking)":
            xtrain.append(extract_features(image_path))

print('Done')
            
xtrain = np.array(xtrain)        


In [ ]:
from sklearn import svm
from sklearn.ensemble import IsolationForest
from sklearn.covariance import EllipticEnvelope # minimum covariance determinant 
from sklearn.neighbors import LocalOutlierFactor

# Standardizing features
scaler = StandardScaler()
features_scaled = scaler.fit_transform(xtrain)

# Train the one-class SVM
clf1svm = svm.OneClassSVM(kernel='rbf', gamma='auto', nu=0.1)
clf1svm.fit(features_scaled)



In [ ]:
clfiso = IsolationForest(n_estimators=100, contamination=0.05)  # contamination parameter depends on expected outlier 
clfiso.fit(features_scaled)


In [ ]:
#clflo = LocalOutlierFactor(n_neighbors=20, contamination=0.1)
#clflo.fit(features_scaled)


In [ ]:
#clfmcd = EllipticEnvelope(support_fraction=1., contamination=0.1)
#clfmcd.fit(features_scaled)

In [ ]:
def predict_1svm_image(img_path):
    features = extract_features(img_path)
    features_scaled = scaler.transform([features])
    prediction = clf1svm.predict(features_scaled)
    return prediction

def predict_iso_image(img_path):
    features = extract_features(img_path)
    features_scaled = scaler.transform([features])
    prediction = clfiso.predict(features_scaled)
    return prediction

#def predict_lo_image(img_path):
 #   features = extract_features(img_path)
    features_scaled = scaler.transform([features])
    #prediction = clflo.predict(features_scaled)
  #  return prediction

#def predict_mcd_image(img_path):
 #   features = extract_features(img_path)
    features_scaled = scaler.transform([features])
    #prediction = clfmcd.predict(features_scaled)
  #  return prediction


In [ ]:
import glob, os
 
sv_in3_label = []
sv_in5_label = []
sv_outlet_label = []

iso_in3_label = []
iso_in5_label = []
iso_outlet_label = []

lo_in3_label = []
lo_in5_label = []
lo_outlet_label = []

mcd_in3_label = []
mcd_in5_label = []
mcd_outlet_label = []


path = "/path/to/private-data/Croped_Pictures/train/Left/*"
for directory_path in glob.glob(path):
    label=directory_path.split("\\")[-1]
    for image_path in glob.glob(os.path.join(directory_path,"*.bmp")):
        svprediction = predict_1svm_image(image_path)
      #  loprediction = predict_lo_image(image_path)
        isoprediction = predict_iso_image(image_path)
       # mcd_prediction = predict_mcd_image(image_path)
        
        
        if label == "Inlet (3 marking)":
            sv_in3_label.append(svprediction)
            iso_in3_label.append(isoprediction)
        #    lo_in3_label.append(loprediction)
         #   mcd_in3_label.append(mcd_prediction)
            

            
        elif label == "Inlet (5 marking)":
            sv_in5_label.append(svprediction)
            iso_in5_label.append(isoprediction)
          #  lo_in5_label.append(loprediction)
           # mcd_in5_label.append(mcd_prediction)
            

            
        elif label == "Outlet":
            sv_outlet_label.append(svprediction)
            iso_outlet_label.append(isoprediction)
           # lo_outlet_label.append(loprediction)
            #mcd_outlet_label.append(mcd_prediction)


In [ ]:
import glob, os
 
val_sv_in3_label = []
val_sv_in5_label = []
val_sv_outlet_label = []
val_sv_others_label = []

val_iso_in3_label = []
val_iso_in5_label = []
val_iso_outlet_label = []
val_iso_others_label = []

val_lo_in3_label = []
val_lo_in5_label = []
val_lo_outlet_label = []
val_lo_others_label = []

val_mcd_in3_label = []
val_mcd_in5_label = []
val_mcd_outlet_label = []
val_mcd_others_label = []


path = "/path/to/private-data/Croped_Pictures/validation/Left/*"
for directory_path in glob.glob(path):
    label=directory_path.split("\\")[-1]
    for image_path in glob.glob(os.path.join(directory_path,"*.bmp")):
        svprediction = predict_1svm_image(image_path)
        #loprediction = predict_lo_image(image_path)
        isoprediction = predict_iso_image(image_path)
        #mcd_prediction = predict_mcd_image(image_path)
        
        
        if label == "Inlet (3 marking)":
            val_sv_in3_label.append(svprediction)
            val_iso_in3_label.append(isoprediction)
         #   val_lo_in3_label.append(loprediction)
          #  val_mcd_in3_label.append(mcd_prediction)
            

            
        elif label == "Inlet (5 marking)":
            val_sv_in5_label.append(svprediction)
            val_iso_in5_label.append(isoprediction)
           # val_lo_in5_label.append(loprediction)
            #val_mcd_in5_label.append(mcd_prediction)
            

            
        elif label == "Outlet":
            val_sv_outlet_label.append(svprediction)
            val_iso_outlet_label.append(isoprediction)
            #val_lo_outlet_label.append(loprediction)
            #val_mcd_outlet_label.append(mcd_prediction)

        elif label == "Outlet":
            val_sv_others_label.append(svprediction)
            val_iso_others_label.append(isoprediction)
            #val_lo_others_label.append(loprediction)
            #val_mcd_others_label.append(mcd_prediction)


In [ ]:
sv_in3_label = np.array(sv_in3_label)
sv_in5_label = np.array(sv_in5_label)
sv_outlet_label = np.array(sv_outlet_label)

iso_in3_label = np.array(iso_in3_label)
iso_in5_label = np.array(iso_in5_label)
iso_outlet_label = np.array(iso_outlet_label)

lo_in3_label = np.array(lo_in3_label)
lo_in5_label = np.array(lo_in5_label)
lo_outlet_label = np.array(lo_outlet_label)

mcd_in3_label = np.array(mcd_in3_label)
mcd_in5_label = np.array(mcd_in5_label)
mcd_outlet_label = np.array(mcd_outlet_label)   


val_sv_others_label = np.array(val_sv_others_label)
val_sv_in5_label = np.array(val_sv_in5_label)
val_sv_outlet_label = np.array(val_sv_outlet_label)

val_iso_in3_label = np.array(val_iso_in3_label)
val_iso_in5_label = np.array(val_iso_in5_label)
val_iso_outlet_label = np.array(val_iso_outlet_label)

val_lo_in3_label = np.array(val_lo_in3_label)
val_lo_in5_label = np.array(val_lo_in5_label)
val_lo_outlet_label = np.array(val_lo_outlet_label)

val_mcd_in3_label = np.array(val_mcd_in3_label)
val_mcd_in5_label = np.array(val_mcd_in5_label)
val_mcd_outlet_label = np.array(val_mcd_outlet_label)   




In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Counting the occurrences of normal (0) and anomaly (1) in each class
normal_counts = np.array([np.count_nonzero(pred == 1) for pred in [sv_in3_label,  sv_in5_label, val_sv_in5_label, sv_outlet_label, val_sv_others_label]])
anomaly_counts = np.array([np.count_nonzero(pred == -1) for pred in [sv_in3_label, sv_in5_label, val_sv_in5_label,  sv_outlet_label, val_sv_others_label]])
confusion_matrix = np.vstack([normal_counts, anomaly_counts])
import pandas as pd

# Creating a DataFrame
class_labels = ['Inlet type -1',  'Inlet type-2','Inlet type -2 (Test)', 'Outlet', 'Unmarked']
df_cm = pd.DataFrame(confusion_matrix, index=['Normal', 'Anomaly'], columns=class_labels)


# Plotting
plt.figure(figsize=(10, 5))
sns.heatmap(df_cm, annot=True, fmt="d", cmap='Blues', cbar=False)
plt.title('Prediction Outcomes using ResNet50 and One-class SVM')
#plt.xlabel('Class')
plt.ylabel('Prediction Type')
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Counting the occurrences of normal (0) and anomaly (1) in each class
normal_counts = np.array([np.count_nonzero(pred == 1) for pred in [iso_in3_label,  iso_in5_label, val_iso_in5_label, iso_outlet_label, val_iso_others_label]])
anomaly_counts = np.array([np.count_nonzero(pred == -1) for pred in [iso_in3_label,  iso_in5_label, val_iso_in5_label, iso_outlet_label, val_iso_others_label]])
confusion_matrix = np.vstack([normal_counts, anomaly_counts])
import pandas as pd

# Creating a DataFrame
class_labels = ['Inlet type -1',  'Inlet type-2','Inlet type -2 (Test)', 'Outlet', 'Unmarked']
df_cm = pd.DataFrame(confusion_matrix, index=['Normal', 'Anomaly'], columns=class_labels)


# Plotting
plt.figure(figsize=(10, 5))
sns.heatmap(df_cm, annot=True, fmt="d", cmap='Blues', cbar=False)
plt.title('Prediction Outcomes using ResNet50 and One-class SVM')
#plt.xlabel('Class')
plt.ylabel('Prediction Type')
plt.show()